# EDA 03 — SWELL-KW Dataset
Exploratory analysis of the SWELL office stress dataset (feature-level only).

Signals (processed): minute-level HR, RMSSD, SCL  
Labels: C=1 → 0 (no stress), C∈{2,3} → 1 (stress)  
Participants: ~25 participants (PP1–PP25)  

**Note**: Raw physiology is in binary .S00 format (Mobi device) — unreadable without proprietary SDK.  
This dataset is used for **cross-dataset evaluation only** (Random Forest, feature-level).

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

SWELL_ROOT = Path('/Users/omarnassar/Documents/Thesis/0_SWELL')
FEAT_PATH  = (SWELL_ROOT / '3 - Feature dataset' / 'per sensor'
              / 'D - Physiology features (HR_HRV_SCL - final).csv')

In [ ]:
# Load feature file
df = pd.read_csv(FEAT_PATH)
df.columns = [c.strip() for c in df.columns]
df.replace(999, np.nan, inplace=True)
print(df.shape)
print(df.head())
print('\nColumns:', list(df.columns))
print('\nCondition counts:')
print(df['C'].value_counts().sort_index())

In [ ]:
# Binary label
df['label'] = df['C'].apply(lambda c: 1 if int(c) in [2, 3] else 0)
print(f"Stress: {df['label'].sum()} / {len(df)} = {100*df['label'].mean():.1f}%")

In [ ]:
# Feature distributions per condition
features = ['HR', 'RMSSD', 'SCL']
cond_map = {1: 'No Stress', 2: 'Time Pressure', 3: 'Email Interruption'}
df_plot = df[df['C'].isin([1, 2, 3])].copy()
df_plot['Condition'] = df_plot['C'].map(cond_map)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = {'No Stress': '#4CAF50', 'Time Pressure': '#F44336', 'Email Interruption': '#FF9800'}

for ax, feat in zip(axes, features):
    for cond, group in df_plot.groupby('Condition'):
        vals = group[feat].dropna()
        ax.hist(vals, bins=30, alpha=0.6, label=cond, color=colors.get(cond))
    ax.set_xlabel(feat)
    ax.set_ylabel('Count')
    ax.set_title(f'{feat} distribution')
    ax.legend(fontsize=7)

fig.suptitle('SWELL: Physiology feature distributions by condition')
plt.tight_layout()
plt.show()

In [ ]:
# Mean features per participant per condition
summary = df_plot.groupby(['PP', 'Condition'])[features].mean().reset_index()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, feat in zip(axes, features):
    for cond, grp in summary.groupby('Condition'):
        ax.scatter(grp['PP'], grp[feat], label=cond, alpha=0.7, s=30, color=colors.get(cond))
    ax.set_xlabel('Participant')
    ax.set_ylabel(f'Mean {feat}')
    ax.set_title(feat)
    ax.legend(fontsize=7)

fig.suptitle('SWELL: Mean features per participant per condition')
plt.tight_layout()
plt.show()

In [ ]:
# Check processed feature windows
processed_path = Path('../data/processed/swell/swell_features.npz')
if processed_path.exists():
    arr = np.load(processed_path, allow_pickle=True)
    print(f'X shape : {arr["X"].shape}  (N windows × 12 features)')
    print(f'y shape : {arr["y"].shape}')
    print(f'Stress  : {arr["y"].sum()} / {len(arr["y"])} = {100*arr["y"].mean():.1f}%')
    print(f'Participants: {len(set(arr["meta"][:, 0]))}')
else:
    print('Run script 03_preprocess_swell.py first.')